In [25]:
from comet_ml import Experiment

In [26]:
import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import random

In [27]:
!pip install comet_ml

In [28]:
class CFG:

    api = "secret"
    project = "oskelly-mlp"
    workspace = "gp5_team1"
    num_workers = 2
    train_batch_size = 64
    test_batch_size = 512

    seed = 22
    comet = True

In [29]:
# Зафиксируем seed для воспроизводимости

def seed_everything(seed):
    random.seed(seed) # фиксируем генератор случайных чисел
    os.environ['PYTHONHASHSEED'] = str(seed) # фиксируем заполнения хешей
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU
    #torch.backends.cudnn.deterministic = True # выбираем только детерминированные алгоритмы (для сверток)
    #torch.backends.cudnn.benchmark = False # фиксируем алгоритм вычисления сверток
seed_everything(CFG.seed)

In [30]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

## Загрузим данные

данные (имеено цены) после этапа eda логарифмированы, будем учитывать это в работе и расчете метрик

In [31]:
X = np.load("X.npy")  # признаки
y = np.load("y.npy")  # целевая переменная цена

In [32]:
pd.Series(y).describe()

,0
count,9268.000000
mean,10.752261
std,0.802238
min,9.210040
25%,10.146796
50%,10.733588
75%,11.326295
max,12.611541


In [33]:
prices = np.exp(y)

pd.Series(prices).describe()

,0
count,9268.000000
mean,64446.734375
std,55786.210938
min,9996.997070
25%,25509.247070
50%,45870.992188
75%,82975.023438
max,300000.906250


Разделим данные на обучающую, валидационную и тестовую


In [34]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=CFG.seed
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=CFG.seed
)

Создадим датасеты и загрузчики данных в формате PyTorch.

В отличие от примера из семинара, здесь мы не загружаем готовый датасет через `datasets.MNIST`, потому что признаки и целевая переменная уже подготовлены заранее и загружены из файлов `X.npy` и `y.npy`.

Поэтому сначала переводим массивы NumPy в тензоры, затем создаём `TensorDataset` и `DataLoader`.

In [35]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [36]:
train_data = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
val_data = torch.utils.data.TensorDataset(X_val_tensor, y_val_tensor)
test_data = torch.utils.data.TensorDataset(X_test_tensor, y_test_tensor)

In [37]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=CFG.train_batch_size, shuffle=True, num_workers=CFG.num_workers)

val_loader = torch.utils.data.DataLoader(val_data, batch_size=CFG.test_batch_size, shuffle=False, num_workers=CFG.num_workers)

test_loader = torch.utils.data.DataLoader(test_data, batch_size=CFG.test_batch_size, shuffle=False, num_workers=CFG.num_workers)

напишем простую MLP-модель - сейчас она будет состоять из двух скрытых полносвязных слоёв и одного выходного слоя, который предсказывает одно число — цену товара

In [38]:
class PriceMLP(nn.Module):
    def __init__(self, input_dim, hidden_1, hidden_2, activation):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, hidden_1)
        self.fc2 = nn.Linear(hidden_1, hidden_2)
        self.fc3 = nn.Linear(hidden_2, 1)

        if activation == "relu":
            self.activation = nn.ReLU()
        elif activation == "tanh":
            self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.fc3(x)
        return x.squeeze()

напишем теперь функции для обучения модели

In [39]:
def train(model, device, train_loader, optimizer, criterion, epoch, logger=None):
    model.train()
    train_loss = 0

    for data, target in tqdm(train_loader):
        data = data.to(device)
        target = target.to(device)

        optimizer.zero_grad()

        output = model(data)
        loss = criterion(output, target)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss = train_loss / len(train_loader)

    print(f"Epoch {epoch}: train_loss = {train_loss:.4f}")

    if logger:
        logger.log_metric("train_loss", train_loss, step=epoch)

    return train_loss

In [40]:
def test(model, device, loader, criterion, epoch, name="val_loss", logger=None):
    model.eval()
    test_loss = 0

    with torch.no_grad():
        for data, target in tqdm(loader):
            data = data.to(device)
            target = target.to(device)

            output = model(data)
            loss = criterion(output, target)

            test_loss += loss.item()

    test_loss = test_loss / len(loader)

    print(f"Epoch {epoch}: {name} = {test_loss:.4f}")

    if logger:
        logger.log_metric(name, test_loss, step=epoch)

    return test_loss

После обучения модели нужно получить предсказания на тестовой выборке

Модель предсказывает логарифм цены, поэтому перед расчётом метрик переводим обратно в рубли

В качестве метрик используем MAE, RMSE, MAPE и R квадрат

In [41]:
def get_predictions(model, device, loader):
    model.eval()

    y_pred = []
    y_true = []

    with torch.no_grad():
        for data, target in tqdm(loader):
            data = data.to(device)

            output = model(data)

            y_pred.extend(output.cpu().numpy())
            y_true.extend(target.numpy())

    y_pred = np.array(y_pred)
    y_true = np.array(y_true)

    return y_true, y_pred

In [42]:
def calculate_price_metrics(y_true_log, y_pred_log):
    y_true_price = np.exp(y_true_log)
    y_pred_price = np.exp(y_pred_log)

    mae = mean_absolute_error(y_true_price, y_pred_price)
    rmse = mean_squared_error(y_true_price, y_pred_price) ** 0.5
    mape = np.mean(np.abs((y_true_price - y_pred_price) / y_true_price)) * 100
    r2 = r2_score(y_true_price, y_pred_price)

    return mae, rmse, mape, r2

создадим массив, гда зададим разные эксперименты, что будем менять - число нейронов в скрытых слоях, функции активации, оптимизаторы и learning rate


In [43]:
experiments = [
    {
        "name": "128-64_relu_adam_lr0001_10ep",
        "hidden_1": 128,
        "hidden_2": 64,
        "activation": "relu",
        "optimizer": optim.Adam,
        "lr": 0.001,
        "epochs": 10
    },
    {
        "name": "256-128_relu_adam_lr0001_10ep",
        "hidden_1": 256,
        "hidden_2": 128,
        "activation": "relu",
        "optimizer": optim.Adam,
        "lr": 0.001,
        "epochs": 10
    },
    {
        "name": "512-256_relu_adam_lr0001_10ep",
        "hidden_1": 512,
        "hidden_2": 256,
        "activation": "relu",
        "optimizer": optim.Adam,
        "lr": 0.001,
        "epochs": 10
    },
    {
        "name": "512-256_relu_adam_lr0005_14ep",
        "hidden_1": 512,
        "hidden_2": 256,
        "activation": "relu",
        "optimizer": optim.Adam,
        "lr": 0.0005,
        "epochs": 14
    },
    {
        "name": "768-384_relu_adam_lr0005_14ep",
        "hidden_1": 768,
        "hidden_2": 384,
        "activation": "relu",
        "optimizer": optim.Adam,
        "lr": 0.0005,
        "epochs": 14
    },

    {
        "name": "256-128_tanh_adam_lr0001_10ep",
        "hidden_1": 256,
        "hidden_2": 128,
        "activation": "tanh",
        "optimizer": optim.Adam,
        "lr": 0.001,
        "epochs": 10
    },
    {
        "name": "512-256_tanh_adam_lr0005_14ep",
        "hidden_1": 512,
        "hidden_2": 256,
        "activation": "tanh",
        "optimizer": optim.Adam,
        "lr": 0.0005,
        "epochs": 14
    },
    {
        "name": "768-384_tanh_adam_lr0005_14ep",
        "hidden_1": 768,
        "hidden_2": 384,
        "activation": "tanh",
        "optimizer": optim.Adam,
        "lr": 0.0005,
        "epochs": 14
    },

    {
        "name": "256-128_relu_sgd_lr001_10ep",
        "hidden_1": 256,
        "hidden_2": 128,
        "activation": "relu",
        "optimizer": optim.SGD,
        "lr": 0.01,
        "epochs": 10
    },
    {
        "name": "512-256_relu_sgd_lr001_14ep",
        "hidden_1": 512,
        "hidden_2": 256,
        "activation": "relu",
        "optimizer": optim.SGD,
        "lr": 0.01,
        "epochs": 14
    },
    {
        "name": "256-128_tanh_sgd_lr001_10ep",
        "hidden_1": 256,
        "hidden_2": 128,
        "activation": "tanh",
        "optimizer": optim.SGD,
        "lr": 0.01,
        "epochs": 10
    },
    {
        "name": "512-256_tanh_sgd_lr001_14ep",
        "hidden_1": 512,
        "hidden_2": 256,
        "activation": "tanh",
        "optimizer": optim.SGD,
        "lr": 0.01,
        "epochs": 14
    },
]

In [44]:
def run_experiment(exp):
    seed_everything(CFG.seed)

    logger = None

    if CFG.comet:
        logger = Experiment(api_key=CFG.api, project_name=CFG.project, workspace=CFG.workspace)

        logger.set_name(exp["name"])

        logger.log_parameters({
            "hidden_1": exp["hidden_1"],
            "hidden_2": exp["hidden_2"],
            "activation": exp["activation"],
            "optimizer": exp["optimizer"].__name__,
            "lr": exp["lr"],
            "epochs": exp["epochs"],
            "batch_size": CFG.train_batch_size,
            "loss": "MSELoss",
            "seed": CFG.seed
        })

    model = PriceMLP(X_train.shape[1], exp["hidden_1"], exp["hidden_2"], exp["activation"]).to(device)

    criterion = nn.MSELoss()
    optimizer = exp["optimizer"](model.parameters(), lr=exp["lr"])

    for epoch in range(1, exp["epochs"] + 1):
        train(model, device, train_loader, optimizer, criterion, epoch, logger)
        test(model, device, val_loader, criterion, epoch, "val_loss", logger)

    y_true, y_pred = get_predictions(model, device, test_loader)
    mae, rmse, mape, r2 = calculate_price_metrics(y_true, y_pred)

    result = {
        "name": exp["name"],
        "hidden_1": exp["hidden_1"],
        "hidden_2": exp["hidden_2"],
        "activation": exp["activation"],
        "optimizer": exp["optimizer"].__name__,
        "lr": exp["lr"],
        "epochs": exp["epochs"],
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "R2": r2
    }

    if logger:
        logger.log_metrics({"MAE": mae, "RMSE": rmse, "MAPE": mape, "R2": r2})

        model_path = exp["name"] + ".pt"
        torch.save(model.state_dict(), model_path)

        logger.log_model("Light-MLP", model_path)
        logger.end()

    return result, model

In [45]:
results = []
models = {}

for exp in experiments:
    result, model = run_experiment(exp)
    results.append(result)
    models[exp["name"]] = model

results_df = pd.DataFrame(results)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/gp5-team1/oskelly-mlp/05cac8b1bcb04eaea1f54d522fc02c70

COMET INFO: Couldn't find a Git repository in '/content' nor in any parent directory. Set `COMET_GIT_DIRECTORY` if your Git Repository is elsewhere.
100%|██████████| 93/93 [00:00<00:00, 104.76it/s]


Epoch 1: train_loss = 25.9192


100%|██████████| 3/3 [00:00<00:00,  9.33it/s]


Epoch 1: val_loss = 1.7699


100%|██████████| 93/93 [00:01<00:00, 76.33it/s]


...

100%|██████████| 93/93 [00:00<00:00, 113.84it/s]


Epoch 14: train_loss = 0.3359


100%|██████████| 3/3 [00:00<00:00, 18.13it/s]


Epoch 14: val_loss = 0.2974


100%|██████████| 4/4 [00:00<00:00, 22.58it/s]
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : 512-256_tanh_sgd_lr001_14ep
COMET INFO:     url                   : https://www.comet.com/gp5-team1/oskelly-mlp/f7d1579c07904530bc1ef568fa95479f
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     MAE             : 24938.09765625
COMET INFO:     MAPE            : 51.57059860229492
COMET INFO:     R2              : 0.490986704826355
COMET INFO:     RMSE            : 39072.8501136019
COMET INFO:     train_loss [14] : (0.3307094096496541, 4.334428774733698)
COMET INFO:     val_loss [14]   : (0.2974245349566142, 0.677970806757609)
COMET INFO:   Others:
COMET INFO:     Name         : 512-256_tanh_sgd_lr0

In [46]:
results_df = pd.DataFrame(results)
results_df

,name,hidden_1,hidden_2,activation,optimizer,lr,epochs,MAE,RMSE,MAPE,R2
0,128-64_relu_adam_lr0001_10ep,128,64,relu,Adam,0.0010,10,29941.181641,51144.449865,60.246311,0.127881
1,256-128_relu_adam_lr0001_10ep,256,128,relu,Adam,0.0010,10,28249.083984,50759.880851,55.044510,0.140947
2,512-256_relu_adam_lr0001_10ep,512,256,relu,Adam,0.0010,10,24753.523438,42906.203934,40.475483,0.386211
3,512-256_relu_adam_lr0005_14ep,512,256,relu,Adam,0.0005,14,23981.755859,41762.341697,41.853439,0.418501
4,768-384_relu_adam_lr0005_14ep,768,384,relu,Adam,0.0005,14,25551.587891,45348.402045,47.026695,0.314349
5,256-128_tanh_adam_lr0001_10ep,256,128,tanh,Adam,0.0010,10,37926.570312,57179.446937,80.684090,-0.090081
6,512-256_tanh_adam_lr0005_14ep,512,256,tanh,Adam,0.0005,14,26925.169922,42990.130216,53.474915,0.383807
7,768-384_tanh_adam_lr0005_14ep,768,384,tanh,Adam,0.0005,14,22951.035156,37207.441836,44.247787,0.538429
8,256-128_relu_sgd_lr001_10ep,256,128,relu,SGD,0.0100,10,25848.974609,39843.307493,54.254837,0.470715
9,512-256_relu_sgd_lr001_14ep,512,256,relu,SGD,0.0100,14,23555.867188,38752.853830,44.310684,0.499290


In [47]:
best_result = results_df.sort_values("R2", ascending=False).iloc[0]
best_result

,7
name,768-384_tanh_adam_lr0005_14ep
hidden_1,768
hidden_2,384
activation,tanh
optimizer,Adam
lr,0.0005
epochs,14
MAE,22951.035156
RMSE,37207.441836
MAPE,44.247787


сохраним таблицу c теми данными, на которых проводились экспериметы

In [48]:
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)

np.save("X_val.npy", X_val)
np.save("y_val.npy", y_val)

np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

## выводы
В этой части были обучены несколько простых MLP-моделей

меняли в экпериментах самые базовые гиперпараметры - размер скрытых слоёв, функция активации, оптимизатор, learning rate и количество эпох

Лучший результат показала модель 768-384_tanh_adam_lr0005_14ep. У неё ошибка MAE составила примерно 22 951 рублей, а R квадрат  0.538. так же видно, что более крупные модели в целом работают лучше маленьких. Также Adam оказался стабильнее SGD